# Training pipeline

## 1. Purpose, scope, and safety

This notebook is the maintained control plane for model training. It selects one official experiment, resolves its current semantic configuration, explains the task-owned data roles, previews deterministic run paths, and presents commands for deliberate terminal use.

With the default controls, top-to-bottom execution is read-only and does not load final tensor datasets, fit normalizers, construct dataloaders or models, initialize W&B, allocate runs, start training, submit jobs, resume runs, execute inference, or generate artifacts.

The final section contains a separate opt-in switch for complete mounted-data validation. Its default is `False`, so ordinary `Run All` remains lightweight. Set the Step 15 switch to `True` only for the expensive read-only check that opens and hashes the mounted ID and OOD datasets and verifies identity, split membership, train-only normalization, dataloaders, and sampler behavior.

Commands shown in later sections are copyable instructions. They are not executable notebook cells.

## 2. Imports and environment

The setup imports configuration, installed notebook support, and display services only. It resolves unified storage paths without querying CUDA or accessing the generated-simulation area. Workflow controls are defined beside the operations they govern.

In [ ]:
from __future__ import annotations

import importlib.metadata
import platform

import pandas as pd
from IPython.display import Markdown
from IPython.display import display as show

from src import common, experiments

PROJECT_ROOT = common.paths.get_project_root()
STORAGE_ROOT = common.paths.get_storage_root()
DATASET_METADATA_ROOT = common.paths.get_dataset_metadata_root()
DATASET_PAYLOAD_ROOT = common.paths.get_dataset_payload_root()
EXPERIMENTS_ROOT = common.paths.get_experiments_root()
PATH_DISPLAY_ROOTS = {
    "project_root": PROJECT_ROOT,
    "storage_root": STORAGE_ROOT,
}

TASK_CONFIG_ROOT = PROJECT_ROOT / "configs" / "tasks"
_EXPERIMENT_PATHS = tuple(
    sorted(path for experiments_root in TASK_CONFIG_ROOT.glob("*/experiments") for path in experiments_root.rglob("*.yaml") if path.is_file())
)
EXPERIMENT_CONFIGS = {path.relative_to(TASK_CONFIG_ROOT).with_suffix("").as_posix(): path for path in _EXPERIMENT_PATHS}
if not EXPERIMENT_CONFIGS:
    message = f"No maintained task-local experiment YAMLs found below {TASK_CONFIG_ROOT}"
    raise RuntimeError(message)

CONTEXT: experiments.notebook_support.NotebookContext | None = None


def package_version(distribution: str) -> str:
    """Return an installed distribution version or an unavailable marker."""
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return "not installed"


def show_tables(*tables: experiments.notebook_support.NotebookTable) -> None:
    """Display ordered support tables through the notebook's pandas surface."""
    for table in tables:
        if table.title is not None:
            show(Markdown(f"#### {table.title}"))
        show(pd.DataFrame(table.rows, columns=table.columns))


def require_context() -> experiments.notebook_support.NotebookContext:
    """Return the prepared context or raise one actionable prerequisite error."""
    if CONTEXT is None:
        message = "Training notebook context is unavailable. Run Step 5: Select, resolve, and validate one experiment."
        raise RuntimeError(message)
    return CONTEXT

In [ ]:
environment_table = pd.DataFrame(
    [
        ("Project root", "$PROJECT_ROOT", "Canonical development-container repository root"),
        ("Storage root", "$STORAGE_ROOT", "Unified numbered scientific lifecycle root"),
        (
            "Dataset metadata root",
            experiments.notebook_support.display_path(DATASET_METADATA_ROOT, **PATH_DISPLAY_ROOTS),
            "Small validated dataset metadata packages",
        ),
        (
            "Dataset payload root",
            experiments.notebook_support.display_path(DATASET_PAYLOAD_ROOT, **PATH_DISPLAY_ROOTS),
            "Final tensor payload location, not opened by this notebook",
        ),
        (
            "Experiments root",
            experiments.notebook_support.display_path(EXPERIMENTS_ROOT, **PATH_DISPLAY_ROOTS),
            "Run-path preview and optional summary inspection",
        ),
        ("Python", platform.python_version(), "Active notebook kernel"),
        ("Project package", package_version("grainlegumes-pino-drying"), "Installed distribution version"),
        ("pandas", package_version("pandas"), "Notebook table rendering"),
    ],
    columns=["Setting", "Value", "Meaning"],
)
show(environment_table)

## 3. Canonical data roles

TaskSpec defines the scientific meaning of ID and OOD roles, the deterministic ID split, train-only normalizer fitting, and which evaluation signals may influence training. Its dataset IDs are fallback defaults only for generic configs that omit an explicit selection.

Each executable experiment YAML is authoritative for its concrete ID and OOD datasets, and explicit YAML values take precedence over the fallback. The selected IDs are displayed from the resolved experiment config immediately after Step 5. The training CLI accepts the recipe path and has no separate dataset-ID arguments.

The ID dataset supplies deterministic training and validation membership. Only training membership fits the normalizer, while ID validation selects checkpoints and drives scheduling and pruning. This selection signal is not an untouched final-test result. OOD membership is diagnostic-only and cannot affect normalization, scheduling, checkpoint selection, or pruning. The saved split and train-fitted normalizer are reused for evaluation, resume, inference, and artifact generation.

## 4. Available experiment configurations

Each maintained production recipe explicitly selects its concrete datasets. The recipes share TaskSpec role semantics and differ in neural-operator architecture and whether physics-informed loss terms are active.

In [ ]:
available_experiments = pd.DataFrame(
    [
        (
            label,
            experiments.notebook_support.display_path(path, **PATH_DISPLAY_ROOTS),
            path.is_file(),
        )
        for label, path in EXPERIMENT_CONFIGS.items()
    ],
    columns=["Experiment", "Configuration", "Exists"],
)
show(available_experiments)

## 5. Select, resolve, and validate one experiment

Choose `CONFIG_PATH` at the top of the cell below. The context is reset before resolution so a failed re-run cannot retain stale state from an earlier kernel execution.

The production loader resolves the selected request. Any current-schema `ConfigError` remains visible as the original failure. The table after the status is derived from the resolved config and shows the recipe-selected ID and OOD datasets.

In [ ]:
CONFIG_PATH = next(iter(EXPERIMENT_CONFIGS.values()))  # Choose one configuration listed in Step 4

CONTEXT = None
CONTEXT = experiments.notebook_support.prepare_notebook_context(CONFIG_PATH)

context_status = pd.DataFrame(
    [
        (
            "Selected configuration",
            experiments.notebook_support.display_path(CONTEXT.config_path, **PATH_DISPLAY_ROOTS),
        ),
        ("Official run name", CONTEXT.official_config["run"]["name"]),
        (
            "Metadata roles validated",
            sum(preview.metadata_validated for preview in CONTEXT.dataset_previews),
        ),
    ],
    columns=["Preparation result", "Value"],
)
show(context_status)

In [ ]:
context = require_context()
resolved_data = context.official_config["data"]
resolved_data_role_table = pd.DataFrame(
    [
        (
            "ID training and validation source",
            resolved_data["train_dataset"],
            "Deterministic training and validation partition",
            "Training fits the normalizer. ID validation selects checkpoints and drives the scheduler and pruning.",
        ),
        (
            "OOD diagnostic source",
            resolved_data["ood_datasets"][0],
            "Separate deterministic diagnostic membership",
            "No normalizer fitting, scheduler decision, checkpoint selection, or pruning influence.",
        ),
    ],
    columns=["Role", "Dataset ID", "Membership", "Training influence"],
)
show(resolved_data_role_table)

## 6. Resolved configuration overview

These ordered tables keep the selected task, architecture, runtime, optimization, physics, evaluation, and W&B settings visible. Installed notebook support formats values already resolved by the production config and task owners. It does not reinterpret their semantics.

In [ ]:
context = require_context()
configuration_tables = experiments.notebook_support.prepare_configuration_tables(context)
show_tables(*configuration_tables)

## 7. Dataset and path overview

This overview validates only the small metadata package. It checks whether each final dataset path exists but never opens or hashes the multi-gigabyte `.pt` payload.

Sample counts and fingerprints remain unavailable when a metadata package is not mounted. Displayed paths use environment-relative names.

In [ ]:
context = require_context()
dataset_table = experiments.notebook_support.prepare_dataset_table(
    context,
    **PATH_DISPLAY_ROOTS,
)
show_tables(dataset_table)

## 8. Run identity and output preview

The production loader owns run-name construction. New leaves use `<architecture>[__<physics>]__<train_dataset>__s<training_seed>[__<run.suffix>]`. UNO architecture includes its resolved scaling sequence and a three-significant-digit mode ratio. PI runs add abbreviated derivative and continuity identity plus both physics weights, supervised runs omit that segment. The exact training-dataset identifier stays separate from the training seed. Optional workflow or experiment context remains final. `run.suffix` must not repeat task, model, architecture, PI physics, dataset, or seed identity. The task remains in the W&B project and local `<output_root>/<task>/runs/` parent, outside the leaf.

Every steady-flow executable request selects `evaluation.objective.id: normalized_group_macro_rmse` explicitly. The loader resolves that exact TaskSpec metric and its lower-is-better direction. Metric declaration order has no model-selection meaning. The objective is dimensionless and is not a percentage. It is an equal macro mean over the pressure and complete velocity groups. Pressure contributes 50 % while the `u` and `v` physical squared errors form the other 50 % through one shared train-fitted velocity-vector scale. Individual normalized and physical field errors remain diagnostics.

The path helper below only computes the intended destination. It does not create a directory, acquire a lock, write `config.yaml`, or initialize W&B.


In [ ]:
context = require_context()
run_preview = experiments.notebook_support.prepare_run_preview_table(
    context,
    **PATH_DISPLAY_ROOTS,
)
show_tables(run_preview)

## 9. Direct-training commands

Select an experiment configuration accepted by the authoritative project loader. Training and Optuna use separate generic interfaces.

Placeholders used in Steps 9–14:

- `<experiment_config>`: path to an experiment YAML
- `<optuna_config>`: path to an Optuna YAML
- `<container_run_dir>`: exact existing run directory visible inside the development container
- `<task>`: registered task ID
- `<run_name>`: exact saved run leaf
- `<log_path>`: queue-log path printed by the launcher

Run training inside the development container:

```bash
cd /workspace/repo

python -m src.experiments.cli.cli_train \
  <experiment_config>
```

Run Optuna through its separate interface:

```bash
cd /workspace/repo

python -m src.experiments.cli.cli_optuna \
  <optuna_config>
```

No `--device` argument uses the YAML policy. `--device cpu` forces CPU. `--device cuda` requires CUDA and never falls back.


## 10. Queued GPU-training commands

Run the generic launcher from the host repository root:

```bash
./scripts/docker_job.sh --queue-gpu auto train <experiment_config>
```

For Optuna:

```bash
./scripts/docker_job.sh --queue-gpu auto optuna <optuna_config>
```

`--queue-gpu auto` selects the least-memory reported GPU non-interactively. Use `--queue-gpu INDEX` for an explicit GPU. In an interactive terminal only, omitting the option prompts for a GPU. The launcher validates the configuration, submits the detached job, prints its labelled host log, and returns immediately. The worker enforces strict CUDA.

Add `--follow` to either command to stream the host log. Ctrl+C stops only log following, the queue job continues.

## 11. Monitor status and logs

The host queue launcher creates the log and prints its exact path with a ready-to-copy command:

```bash
tail -F <log_path>
```

A direct run writes the same persistent line-oriented output to its foreground terminal. Startup identifies the resolved run, task-scoped W&B project, architecture, data roles, loss composition, objective, evaluation cadences, device, determinism, and output path. Every completed optimizer epoch prints loss components, weights, learning rate, timing, throughput, elapsed time, ETA, and last-checkpoint publication. Due ID, OOD, physics, best-checkpoint, scheduler-change, Optuna, selected-model, and failure phases have their own lines. No per-batch output is enabled by default. These diagnostics remain complete when W&B is offline or disabled.

All normal interval phases use completed epochs only: an event is due on an interval multiple or the terminal target. Production/study ID, OOD, and physics histories therefore run at `5, 10, ..., 600`. Optuna evaluates ID every epoch for pruning while OOD and physics remain at interval five for diagnostic cost. Scheduling, selection, and Optuna reuse that one ID validation result. After training, final ID validation, OOD diagnostic, and physics values come from a reloaded `best_checkpoint.pt` and remain separate from terminal history.

The expected run directory is shown in Step 8. Inspect a local lifecycle summary without loading checkpoints:

```bash
python -m json.tool \
  "<container_run_dir>/summary.json"
```

Production tracking is online and fail-closed. Authentication uses `WANDB_API_KEY` non-interactively. Explicit `offline` and `disabled` modes remain available through valid configurations.

Online history uses the automatic personal-workspace categories `Overview`, `Accuracy`, `Physics`, and `Diagnostics`. Selected scalars and tables are summarized. Datasets and checkpoints are not uploaded automatically. Normal run tags contain the model variant, Optuna adds `optuna`, and manual tags survive resume.


## 12. Resume an existing run

Resume requires a semantically compatible config and the exact existing run directory. Use this generic template inside the development container:

```bash
cd /workspace/repo

python -m src.experiments.cli.cli_train \
  <experiment_config> \
  --resume "<container_run_dir>"
```

The same operation can be queued from the host repository root:

```bash
./scripts/docker_job.sh train <experiment_config> --resume "<container_run_dir>"
```

`training.epochs` must exceed completed progress to perform more work. Apart from that extension, resolved paths, and `run.device`, the resolved task, data, model, loss, objective, optimizer, scheduler, and tracking settings must remain compatible.

Resume restores the complete continuation state from `last_checkpoint.pt`, including model and optimization state, randomness, loader and sampler state, split, normalizer, and W&B identity. It continues at the next genuine completed epoch without replaying prior training, ID, OOD, or physics events. An extended target adds only later interval events plus its new terminal event. `best_checkpoint.pt` remains the selected source for evaluation, inference, artifacts, and runtime comparison unless a later ID objective improves it.

## 13. Inspect completed-run files and summary

The displayed inventory covers `config.yaml`, `summary.json`, `split_indices.pt`, `normalizer.pt`, `best_checkpoint.pt`, `last_checkpoint.pt`, local `wandb/` state, `analysis/id/`, `analysis/ood/<dataset>/`, and artifact provenance with case payloads. Checkpoint roles are defined in Step 12.

Set `INSPECT_RUN_DIR` at the top of the cell below for lightweight local inspection. The optional branch reads `summary.json` and checks expected paths without loading tensors, datasets, inference contexts, or artifacts.

In [ ]:
INSPECT_RUN_DIR = None  # Completed run path, or None to skip

context = require_context()
run_output_inventory = experiments.notebook_support.prepare_run_output_inventory_table()
show_tables(run_output_inventory)

if INSPECT_RUN_DIR is None:
    show(Markdown("Completed-run inspection skipped. Set `INSPECT_RUN_DIR` in Step 2 to an explicit run directory and rerun this cell."))
else:
    inspection = experiments.notebook_support.prepare_run_inspection(
        INSPECT_RUN_DIR,
        ood_dataset_id=context.official_config["data"]["ood_datasets"][0],
    )
    inspection_tables = experiments.notebook_support.prepare_run_inspection_tables(
        inspection,
        **PATH_DISPLAY_ROOTS,
    )
    show_tables(*inspection_tables)

## 14. Continue with evaluation and artifacts

Use `eval_single_model.ipynb` for one completed model and `eval_comparison_models.ipynb` for several models. Both use `best_checkpoint.pt` with the saved split, train-fitted normalizer, and the exact `normalized_group_macro_rmse` identity. Incompatible prior-objective artifacts are rejected rather than converted.

Use this generic artifact command inside the development container:

```bash
cd /workspace/repo

python -m src.experiments.cli.cli_build_artifacts \
  --task <task> \
  --run-name <run_name>
```

The same interface is available from the host repository root:

```bash
./scripts/docker_job.sh artifacts --task <task> --run-name <run_name>
```

The evaluation notebooks are load-only and do not select an inference device. Artifact generation supports CPU and CUDA, the artifact CLI defaults to `auto` and records the resolved device. For initial GPU artifact generation, use `--device cuda`. Add `--rebuild` only when deliberately replacing already existing artifacts, for example when regenerating CPU-produced runtime evidence on CUDA. Strict CUDA fails instead of falling back to CPU.

`runtime_comparison.json` uses `best_checkpoint.pt`, the completed run's saved `data.batch_size`, one excluded warm-up, evaluation and inference mode, and CUDA synchronization when applicable. Observed loader-batch forward time is attributed equally across its cases. Speedup is COMSOL solve seconds divided by the amortized neural forward seconds for the matching case identity.


## 15. Optional full data-pipeline validation

This explicit opt-in check reads and hashes both complete mounted tensor datasets selected by the resolved experiment. It verifies dataset identity, split membership, train-only normalization, dataloaders, and sampler state, then displays compact validation tables. It is read-only but can require substantial time and memory.

Change the first line of the cell below from `False` to `True` and run this section only when the complete pre-training check is required.

In [ ]:
RUN_FULL_DATA_VALIDATION = False  # True only for the expensive read-only check

if not RUN_FULL_DATA_VALIDATION:
    show(Markdown("Full data-pipeline validation skipped. Set `RUN_FULL_DATA_VALIDATION = True` above and rerun this section."))
else:
    context = require_context()
    full_validation = experiments.validation.data_pipeline.validate_full_data_pipeline(context.official_config)
    validation_presentation = experiments.notebook_support.prepare_validation_presentation(full_validation)
    show_tables(*validation_presentation.tables)
    show(Markdown(validation_presentation.conclusion))